In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / "pass.env"
loaded = load_dotenv(dotenv_path=env_path, override=True)
print(f"¿Archivo encontrado y cargado?: {loaded}")

PG_CONNECTION_STRING = os.getenv("PG_CONNECTION_STRING")
if not PG_CONNECTION_STRING:
    raise ValueError("Falta PG_CONNECTION_STRING en pass.env")

print("Variables cargadas correctamente.")


¿Archivo encontrado y cargado?: True
Variables cargadas correctamente.


In [2]:
import psycopg2

conn = psycopg2.connect(PG_CONNECTION_STRING)
cur = conn.cursor()

cur.execute("SELECT version();")
print(" Conectado a PostgreSQL")
print(cur.fetchone()[0])


 Conectado a PostgreSQL
PostgreSQL 18.6 (c5250a2) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


---
## 2. Preparación: Faker y semilla reproducible

Si `faker` no está instalado, este notebook cae a listas fijas de respaldo (igual que hizo el profesor en la Semana 3), así que puede correr aunque no se haya instalado el paquete.

In [3]:
%pip install -q faker


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import random
import json
import hashlib
from datetime import date, datetime, timedelta

try:
    from faker import Faker
    fake = Faker("es_CO")
    fake.seed_instance(42)
    FAKER_DISPONIBLE = True
except ImportError:
    FAKER_DISPONIBLE = False
    print(" Faker no está instalado; usamos listas fijas de nombres de respaldo.")

random.seed(42)

NOMBRES_RESPALDO = ["Laura", "Andres", "Valentina", "Juan", "Camila", "Santiago",
                    "Mariana", "David", "Sofia", "Daniel", "Paula", "Felipe"]
APELLIDOS_RESPALDO = ["Rodriguez", "Gomez", "Torres", "Ramirez", "Herrera", "Diaz",
                      "Lopez", "Castro", "Martinez", "Perez", "Ortiz", "Vargas"]
MUNICIPIOS_RESPALDO = ["Popayán", "Cali", "Pasto", "Ibagué", "Neiva", "Villavicencio",
                       "Montería", "Cúcuta", "Bucaramanga", "Manizales"]

def nombre_apellido():
    if FAKER_DISPONIBLE:
        return fake.first_name(), fake.last_name()
    return random.choice(NOMBRES_RESPALDO), random.choice(APELLIDOS_RESPALDO)

def municipio():
    if FAKER_DISPONIBLE:
        return fake.city()
    return random.choice(MUNICIPIOS_RESPALDO)

def direccion():
    if FAKER_DISPONIBLE:
        return fake.street_address()
    return f"Calle {random.randint(1,150)} # {random.randint(1,99)}-{random.randint(1,99)}"

def hash_password(password):
    # Solo para datos sintéticos de prueba; en producción se usa bcrypt/argon2, nunca sha256 plano.
    return hashlib.sha256(password.encode()).hexdigest()

documentos_usados = set()

def generar_documento():
    while True:
        doc = random.randint(1_000_000_000, 1_199_999_999)
        if doc not in documentos_usados:
            documentos_usados.add(doc)
            return doc

print(" Utilidades de generación listas.")


 Utilidades de generación listas.


---
## 3. Catálogos de referencia (CIE-10, LOINC, medicamentos)

Los códigos **CIE-10** (diagnóstico) y **LOINC** (signos vitales) usados abajo son códigos reales de esas terminologías internacionales — lo sintético es qué paciente recibe cuál, no el código en sí mismo.

In [5]:
DIAGNOSTICOS_CIE10 = [
    ("J00", "Rinofaringitis aguda (resfriado común)"),
    ("I10", "Hipertensión esencial"),
    ("E11", "Diabetes mellitus tipo 2"),
    ("M54", "Dorsalgia"),
    ("R51", "Cefalea"),
    ("Z00", "Examen general (control de rutina)"),
    ("A09", "Diarrea y gastroenteritis de presunto origen infeccioso"),
    ("J45", "Asma"),
    ("R10", "Dolor abdominal y pélvico"),
    ("S72", "Fractura del fémur"),
]

# Códigos LOINC reales para signos vitales, con generador de valor y unidad.
SIGNOS_VITALES_LOINC = [
    ("8310-5", "Temperatura corporal", lambda: round(random.uniform(36.0, 39.5), 1), "Cel"),
    ("8867-4", "Frecuencia cardiaca", lambda: random.randint(55, 130), "lpm"),
    ("9279-1", "Frecuencia respiratoria", lambda: random.randint(12, 28), "resp/min"),
    ("8480-6", "Presión arterial sistólica", lambda: random.randint(90, 160), "mmHg"),
    ("8462-4", "Presión arterial diastólica", lambda: random.randint(55, 100), "mmHg"),
    ("59408-5", "Saturación de oxígeno (pulsioximetría)", lambda: random.randint(88, 100), "%"),
]

# Catálogo de medicamentos. Códigos CUM y registros INVIMA con FORMATO real pero valores
# ficticios (no corresponden a medicamentos reales registrados con ese número exacto).
MEDICAMENTOS = [
    ("19000001-1", "Acetaminofén 500mg", "Acetaminofén", "500 mg", "Tableta", "INVIMA 2015M-000101", "Vigente", 250.0),
    ("19000002-1", "Ibuprofeno 400mg", "Ibuprofeno", "400 mg", "Tableta", "INVIMA 2016M-000102", "Vigente", 300.0),
    ("19000003-1", "Amoxicilina 500mg", "Amoxicilina", "500 mg", "Cápsula", "INVIMA 2014M-000103", "Vigente", 450.0),
    ("19000004-1", "Losartán 50mg", "Losartán potásico", "50 mg", "Tableta", "INVIMA 2013M-000104", "Vigente", 320.0),
    ("19000005-1", "Metformina 850mg", "Metformina clorhidrato", "850 mg", "Tableta", "INVIMA 2013M-000105", "Vigente", 280.0),
    ("19000006-1", "Omeprazol 20mg", "Omeprazol", "20 mg", "Cápsula", "INVIMA 2012M-000106", "Vigente", 200.0),
    ("19000007-1", "Loratadina 10mg", "Loratadina", "10 mg", "Tableta", "INVIMA 2011M-000107", "Vigente", 150.0),
    ("19000008-1", "Ácido acetilsalicílico 100mg", "Ácido acetilsalicílico", "100 mg", "Tableta", "INVIMA 2010M-000108", "Vigente", 100.0),
    ("19000009-1", "Salbutamol inhalador 100mcg", "Salbutamol", "100 mcg/dosis", "Inhalador", "INVIMA 2017M-000109", "Vigente", 18000.0),
    ("19000010-1", "Dexametasona 4mg/ml inyectable", "Dexametasona", "4 mg/ml", "Ampolla", "INVIMA 2015M-000110", "Vigente", 900.0),
    ("19000011-1", "Solución salina 0.9% 1000ml", "Cloruro de sodio", "0.9%", "Bolsa IV", "INVIMA 2009M-000111", "Vigente", 5200.0),
    ("19000012-1", "Insulina glargina 100U/ml", "Insulina glargina", "100 U/ml", "Vial", "INVIMA 2018M-000112", "Vigente", 45000.0),
]

SINTOMAS_PREVIOS = [
    "Dolor torácico intenso", "Dificultad para respirar", "Fiebre alta persistente",
    "Dolor abdominal severo", "Sangrado vaginal en gestante", "Trauma por caída",
    "Convulsiones", "Pérdida de conciencia transitoria",
]

TIPOS_ANTECEDENTE = [
    ("patologico", "Hipertensión arterial diagnosticada"),
    ("patologico", "Diabetes mellitus tipo 2"),
    ("quirurgico", "Apendicectomía"),
    ("alergico", "Alergia a penicilina"),
    ("familiar", "Antecedente familiar de enfermedad cardiovascular"),
    ("toxico", "Tabaquismo activo"),
]

print(f" {len(DIAGNOSTICOS_CIE10)} diagnósticos CIE-10, {len(SIGNOS_VITALES_LOINC)} signos vitales LOINC, "
      f"{len(MEDICAMENTOS)} medicamentos listos.")


 10 diagnósticos CIE-10, 6 signos vitales LOINC, 12 medicamentos listos.


---
## 4. Limpieza de datos de ejecuciones anteriores

Igual que hizo el profesor con `DELETE FROM consultas` antes de recargar: esto permite volver a correr el notebook las veces que se quiera sin chocar con llaves duplicadas. **No borra `roles`** (esas 4 filas ya las crea `create_roles.ipynb`) ni la estructura de las tablas.

In [6]:
conn.rollback()

# Primero se limpian las auto-referencias (deleted_by) para que el DELETE no choque con el FK.
for tabla in ["usuarios", "pacientes", "antecedentes", "reportes_previos", "encuentros",
              "observaciones", "prescripciones", "facturas", "factura_detalle"]:
    cur.execute(f"UPDATE {tabla} SET deleted_by = NULL")

for tabla in ["auditoria_cambios", "factura_detalle", "facturas", "prescripciones", "medicamentos",
              "observaciones", "encuentros", "reportes_previos", "antecedentes", "pacientes", "usuarios"]:
    cur.execute(f"DELETE FROM {tabla}")

conn.commit()
print(" Datos de ejecuciones anteriores eliminados (roles se conservan).")


 Datos de ejecuciones anteriores eliminados (roles se conservan).


---
## 5. Usuarios de personal (Admin, Médico, Administrativo)

Se toma el `id_rol` real desde la tabla `roles` (no se hardcodea), tal como haría un servicio de integración real que no puede asumir que los IDs autogenerados serán siempre 1, 2, 3…

In [7]:
cur.execute("SELECT id_rol, nombre FROM roles")
ROLES = {nombre: id_rol for id_rol, nombre in cur.fetchall()}
print("Roles disponibles:", ROLES)

if len(ROLES) < 4:
    raise RuntimeError("Faltan roles en la tabla roles. Corre primero create_roles.ipynb.")


Roles disponibles: {'Admin': 1, 'Medico': 2, 'Administrativo': 3, 'Paciente': 4}


In [8]:
def crear_usuario(rol_nombre, nombres, apellidos, prefijo_username):
    doc = generar_documento()
    username = f"{prefijo_username}.{apellidos.lower()}{doc % 1000}"
    return {
        "numero_documento_usuario": doc,
        "id_rol": ROLES[rol_nombre],
        "username": username,
        "password_hash": hash_password("Cambiar123*"),
        "nombres": nombres,
        "apellidos": apellidos,
        "email": f"{username}@saluddigital.test",
        "telefono": f"3{random.randint(100000000, 199999999)}",
    }

usuarios_staff = []

nombres, apellidos = nombre_apellido()
usuarios_staff.append(crear_usuario("Admin", nombres, apellidos, "admin"))

for _ in range(4):
    nombres, apellidos = nombre_apellido()
    usuarios_staff.append(crear_usuario("Medico", nombres, apellidos, "dr"))

for _ in range(2):
    nombres, apellidos = nombre_apellido()
    usuarios_staff.append(crear_usuario("Administrativo", nombres, apellidos, "adm"))

for u in usuarios_staff:
    cur.execute("""
        INSERT INTO usuarios (numero_documento_usuario, id_rol, username, password_hash,
                               nombres, apellidos, email, telefono)
        VALUES (%(numero_documento_usuario)s, %(id_rol)s, %(username)s, %(password_hash)s,
                %(nombres)s, %(apellidos)s, %(email)s, %(telefono)s)
    """, u)
conn.commit()

usuario_admin = usuarios_staff[0]["numero_documento_usuario"]
usuarios_medicos = [u["numero_documento_usuario"] for u in usuarios_staff if u["id_rol"] == ROLES["Medico"]]
usuarios_administrativos = [u["numero_documento_usuario"] for u in usuarios_staff if u["id_rol"] == ROLES["Administrativo"]]

print(f" {len(usuarios_staff)} usuarios de personal creados "
      f"(1 Admin, {len(usuarios_medicos)} Médicos, {len(usuarios_administrativos)} Administrativos).")


 7 usuarios de personal creados (1 Admin, 4 Médicos, 2 Administrativos).


---
## 6. Pacientes

El 75% recibe cuenta de usuario propia (rol `Paciente`, para el portal de solo lectura); el 25% restante queda registrado solo por el personal clínico (`id_usuario` nulo), como pasaría con un paciente atendido en urgencias sin acceso a internet — el campo es `UNIQUE` y opcional, así que el esquema ya soporta ambos casos.

In [9]:
N_PACIENTES = 25
pacientes_data = []

for _ in range(N_PACIENTES):
    nombres, apellidos = nombre_apellido()
    doc_paciente = generar_documento()
    fecha_nac = date(1950, 1, 1) + timedelta(days=random.randint(0, 27000))
    edad = 2026 - fecha_nac.year
    tipo_documento = "TI" if edad < 18 else random.choices(["CC", "CE"], weights=[0.9, 0.1])[0]
    sexo = random.choice(["F", "M"])
    zona = random.choices(["urbana", "rural_dispersa"], weights=[0.6, 0.4])[0]
    tiene_cuenta = random.random() < 0.75

    id_usuario = None
    if tiene_cuenta:
        username = f"paciente.{apellidos.lower()}{doc_paciente % 1000}"
        cur.execute("""
            INSERT INTO usuarios (numero_documento_usuario, id_rol, username, password_hash,
                                   nombres, apellidos, email, telefono)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (doc_paciente, ROLES["Paciente"], username, hash_password("Paciente123*"),
              nombres, apellidos, f"{username}@correo.test",
              f"3{random.randint(100000000, 199999999)}"))
        id_usuario = doc_paciente

    pacientes_data.append({
        "numero_documento_paciente": doc_paciente,
        "id_usuario": id_usuario,
        "tipo_documento": tipo_documento,
        "nombres": nombres,
        "apellidos": apellidos,
        "fecha_nacimiento": fecha_nac,
        "sexo": sexo,
        "telefono": f"3{random.randint(100000000, 199999999)}",
        "direccion": direccion(),
        "municipio_residencia": municipio(),
        "zona_residencia": zona,
    })

for p in pacientes_data:
    cur.execute("""
        INSERT INTO pacientes (numero_documento_paciente, id_usuario, tipo_documento, nombres,
                                apellidos, fecha_nacimiento, sexo, telefono, direccion,
                                municipio_residencia, zona_residencia)
        VALUES (%(numero_documento_paciente)s, %(id_usuario)s, %(tipo_documento)s, %(nombres)s,
                %(apellidos)s, %(fecha_nacimiento)s, %(sexo)s, %(telefono)s, %(direccion)s,
                %(municipio_residencia)s, %(zona_residencia)s)
    """, p)
conn.commit()

con_cuenta = sum(1 for p in pacientes_data if p["id_usuario"])
print(f" {len(pacientes_data)} pacientes creados ({con_cuenta} con cuenta de portal, "
      f"{len(pacientes_data) - con_cuenta} sin cuenta).")


 25 pacientes creados (21 con cuenta de portal, 4 sin cuenta).


---
## 7. Antecedentes

Alrededor del 60% de los pacientes tiene 1 o 2 antecedentes registrados por un médico.

In [ ]:
antecedentes_creados = 0
for p in pacientes_data:
    if random.random() < 0.6:
        for _ in range(random.randint(1, 2)):
            tipo, descripcion = random.choice(TIPOS_ANTECEDENTE)
            cur.execute("""
                INSERT INTO antecedentes (id_paciente, tipo, descripcion, registrado_por)
                VALUES (%s, %s, %s, %s)
            """, (p["numero_documento_paciente"], tipo, descripcion, random.choice(usuarios_medicos)))
            antecedentes_creados += 1
conn.commit()
print(f" {antecedentes_creados} antecedentes registrados.")


---
## 8. Reportes previos (telemedicina / zona rural dispersa)

Se generan sobre todo para pacientes en `zona_residencia = 'rural_dispersa'`, que es el caso de uso que le da sentido a esta tabla (distancia y tiempo de desplazamiento hasta el centro de salud).

In [10]:
reportes_creados = 0
for p in pacientes_data:
    if p["zona_residencia"] == "rural_dispersa" and random.random() < 0.7:
        fecha_reporte = datetime(2026, random.randint(1, 9), random.randint(1, 28),
                                  random.randint(0, 23), random.randint(0, 59))
        signos_alarma = random.random() < 0.4
        cur.execute("""
            INSERT INTO reportes_previos (id_paciente, fecha_hora_reporte, sintoma_principal,
                                           inicio_sintomas, evolucion, signos_alarma_presentes,
                                           descripcion_signos_alarma, ubicacion_aproximada,
                                           municipio_origen, distancia_aproximada_km,
                                           tiempo_desplazamiento_min, orientacion_inicial,
                                           registrado_por)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            p["numero_documento_paciente"], fecha_reporte, random.choice(SINTOMAS_PREVIOS),
            fecha_reporte - timedelta(hours=random.randint(1, 48)),
            "Síntomas en aumento progresivo" if signos_alarma else "Estable desde el inicio",
            signos_alarma,
            "Dificultad respiratoria y palidez" if signos_alarma else None,
            f"Vereda cercana a {p['municipio_residencia']}", p["municipio_residencia"],
            round(random.uniform(5, 120), 1), random.randint(20, 300),
            "Se orienta traslado a centro de salud más cercano",
            random.choice(usuarios_administrativos + usuarios_medicos),
        ))
        reportes_creados += 1
conn.commit()
print(f" {reportes_creados} reportes previos (telemedicina rural) registrados.")


 4 reportes previos (telemedicina rural) registrados.


---
## 9. Encuentros (con triage integrado)

Cada paciente tiene entre 1 y 3 encuentros. El `nivel_triage` sigue una distribución realista (la mayoría de pacientes de urgencias caen en triage 3-4, pocos en 1-2).

In [11]:
ESTADOS_ENCUENTRO = ["en_triage", "en_atencion", "en_observacion", "finalizado"]

encuentros_creados = []
for p in pacientes_data:
    for _ in range(random.randint(1, 3)):
        ingreso = datetime(2026, random.randint(1, 9), random.randint(1, 28),
                            random.randint(0, 23), random.randint(0, 59))
        estado = random.choices(ESTADOS_ENCUENTRO, weights=[0.05, 0.15, 0.2, 0.6])[0]
        fin = ingreso + timedelta(hours=random.randint(1, 12)) if estado == "finalizado" else None
        nivel_triage = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.15, 0.35, 0.30, 0.15])[0]
        dolor = random.randint(0, 10)
        codigo_cie10, descripcion_dx = random.choice(DIAGNOSTICOS_CIE10)
        medico_triage = random.choice(usuarios_medicos)
        creado_por = random.choice(usuarios_administrativos + usuarios_medicos)

        cur.execute("""
            INSERT INTO encuentros (id_paciente, fecha_hora_ingreso, fecha_hora_fin, tipo_encuentro,
                                     servicio, estado, motivo_consulta, observaciones_generales,
                                     nivel_triage, fecha_hora_triage, dolor_escala,
                                     observaciones_triage, clasificado_por, clasificacion_automatica,
                                     creado_por)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            RETURNING id_encuentro
        """, (
            p["numero_documento_paciente"], ingreso, fin, "urgencia", "URGENCIAS", estado,
            f"{codigo_cie10} - {descripcion_dx}",
            "Paciente estable al ingreso" if nivel_triage >= 4 else "Paciente requiere atención prioritaria",
            nivel_triage, ingreso + timedelta(minutes=random.randint(2, 20)), dolor,
            "Clasificación según escala de dolor y signos vitales", medico_triage, False, creado_por,
        ))
        id_encuentro = cur.fetchone()[0]
        encuentros_creados.append({
            "id_encuentro": id_encuentro,
            "id_paciente": p["numero_documento_paciente"],
            "ingreso": ingreso,
            "estado": estado,
            "medico": medico_triage,
        })
conn.commit()
print(f" {len(encuentros_creados)} encuentros clínicos creados.")


 51 encuentros clínicos creados.


---
## 10. Observaciones (signos vitales, con código LOINC)

Cada encuentro recibe el panel completo de 6 signos vitales, todos con su código LOINC real.

In [12]:
observaciones_creadas = 0
for e in encuentros_creados:
    momento = e["ingreso"] + timedelta(minutes=random.randint(5, 30))
    for codigo, nombre_signo, generador, unidad in SIGNOS_VITALES_LOINC:
        valor = generador()
        cur.execute("""
            INSERT INTO observaciones (id_encuentro, tipo_observacion, codigo_loinc, nombre,
                                        valor_numerico, unidad, fecha_hora_observacion,
                                        registrado_por)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (e["id_encuentro"], "signo_vital", codigo, nombre_signo, valor, unidad,
              momento, e["medico"]))
        observaciones_creadas += 1
conn.commit()
print(f" {observaciones_creadas} observaciones (signos vitales) registradas.")


 306 observaciones (signos vitales) registradas.


---
## 11. Medicamentos (catálogo)

Se carga una sola vez el catálogo completo, con `ON CONFLICT DO NOTHING` para que sea seguro re-ejecutar.

In [13]:
for cod, nom, principio, conc, forma, registro, estado_cum, precio in MEDICAMENTOS:
    cur.execute("""
        INSERT INTO medicamentos (codigo_cum, nombre, principio_activo, concentracion,
                                   forma_farmaceutica, registro_sanitario, estado_cum, precio_unitario)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (codigo_cum) DO NOTHING
    """, (cod, nom, principio, conc, forma, registro, estado_cum, precio))
conn.commit()
print(f" {len(MEDICAMENTOS)} medicamentos cargados en el catálogo.")


 12 medicamentos cargados en el catálogo.


---
## 12. Prescripciones

Aproximadamente el 70% de los encuentros generan 1 o 2 prescripciones, hechas por el mismo médico que clasificó el triage.

In [14]:
prescripciones_creadas = []
for e in encuentros_creados:
    if random.random() < 0.7:
        for _ in range(random.randint(1, 2)):
            cod, nom, principio, conc, forma, registro, estado_cum, precio = random.choice(MEDICAMENTOS)
            cantidad = random.randint(1, 3)
            estado_presc = random.choices(["activa", "dispensada", "anulada"], weights=[0.5, 0.4, 0.1])[0]
            cur.execute("""
                INSERT INTO prescripciones (id_encuentro, codigo_cum, dosis, frecuencia,
                                             via_administracion, cantidad, prescrito_por, estado)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                RETURNING id_prescripcion
            """, (e["id_encuentro"], cod, "1 unidad", "Cada 8 horas", "Oral", cantidad,
                  e["medico"], estado_presc))
            id_prescripcion = cur.fetchone()[0]
            prescripciones_creadas.append({
                "id_prescripcion": id_prescripcion,
                "id_encuentro": e["id_encuentro"],
                "codigo_cum": cod,
                "cantidad": cantidad,
                "precio": float(precio),
            })
conn.commit()
print(f" {len(prescripciones_creadas)} prescripciones registradas.")


 61 prescripciones registradas.


---
## 13. Facturas y factura_detalle

El 80% de los encuentros se factura. Cada factura incluye la línea de "Consulta de urgencias" más una línea por cada medicamento prescrito en ese encuentro, y el `total` de la factura se calcula como la suma exacta de sus líneas de detalle (para que cuadre en cualquier auditoría).

In [15]:
facturas_creadas = 0
detalle_creados = 0
contador_factura = 1
VALOR_CONSULTA = 45000.0

for e in encuentros_creados:
    if random.random() < 0.8:
        numero_factura = f"FE-2026-{contador_factura:05d}"
        contador_factura += 1
        administrativo = random.choice(usuarios_administrativos)
        fecha_emision = e["ingreso"] + timedelta(hours=random.randint(1, 24))

        detalles = [{"concepto": "Consulta de urgencias", "cantidad": 1,
                     "valor_unitario": VALOR_CONSULTA, "id_prescripcion": None}]
        for pr in prescripciones_creadas:
            if pr["id_encuentro"] == e["id_encuentro"]:
                detalles.append({
                    "concepto": f"Medicamento {pr['codigo_cum']}",
                    "cantidad": pr["cantidad"],
                    "valor_unitario": pr["precio"],
                    "id_prescripcion": pr["id_prescripcion"],
                })

        total = sum(d["cantidad"] * d["valor_unitario"] for d in detalles)
        estado_factura = random.choices(["pendiente", "pagada", "anulada"], weights=[0.3, 0.6, 0.1])[0]

        cur.execute("""
            INSERT INTO facturas (id_paciente, id_encuentro, numero_factura, fecha_emision,
                                   concepto, total, estado, creado_por)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            RETURNING id_factura
        """, (e["id_paciente"], e["id_encuentro"], numero_factura, fecha_emision,
              "Atención de urgencias", total, estado_factura, administrativo))
        id_factura = cur.fetchone()[0]
        facturas_creadas += 1

        for d in detalles:
            valor_total_linea = d["cantidad"] * d["valor_unitario"]
            cur.execute("""
                INSERT INTO factura_detalle (id_factura, id_prescripcion, concepto, cantidad,
                                              valor_unitario, valor_total)
                VALUES (%s, %s, %s, %s, %s, %s)
            """, (id_factura, d["id_prescripcion"], d["concepto"], d["cantidad"],
                  d["valor_unitario"], valor_total_linea))
            detalle_creados += 1

conn.commit()
print(f" {facturas_creadas} facturas y {detalle_creados} líneas de detalle creadas.")


 40 facturas y 90 líneas de detalle creadas.


---
## 14. Verificación final

Conteo de filas por tabla, para confirmar que todo quedó cargado.

In [16]:
tablas = ["roles", "usuarios", "pacientes", "antecedentes", "reportes_previos", "encuentros",
          "observaciones", "medicamentos", "prescripciones", "facturas", "factura_detalle"]

print("Resumen de registros cargados:")
for t in tablas:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t}: {cur.fetchone()[0]}")


Resumen de registros cargados:
  roles: 4
  usuarios: 28
  pacientes: 25
  antecedentes: 0
  reportes_previos: 4
  encuentros: 51
  observaciones: 306
  medicamentos: 12
  prescripciones: 61
  facturas: 40
  factura_detalle: 90


In [17]:
cur.close()
conn.close()
print(" Conexión cerrada. Datos sintéticos cargados correctamente.")


 Conexión cerrada. Datos sintéticos cargados correctamente.
